In [0]:
# Cell1 — Setup & UDFs inline
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, StringType
from pyspark.sql.types import IntegerType

# UDF: extract unit from event_distance_length
def get_event_unit(val):
    if val is None: return None
    val = val.lower().strip()
    if val.endswith("km"): return "km"
    elif val.endswith("mi"): return "mi"
    elif val.endswith("h"):  return "h"
    elif val.endswith("d"):  return "d"
    return None

get_event_unit_udf = F.udf(get_event_unit, StringType())

# UDF: extract unit from athlete_performance
def get_performance_unit(val):
    if val is None: return None
    if ":" in val.strip(): return "h"
    try:
        float(val.strip())
        return "km"
    except ValueError:
        return None

get_performance_unit_udf = F.udf(get_performance_unit, StringType())

# UDF: convert HH:MM:SS to seconds
def time_to_seconds(time_str):
    if time_str is None: return None
    try:
        s = time_str.strip()
        extra_days = 0
        if "day" in s:
            parts = s.split(",")
            extra_days = int(parts[0].strip().split(" ")[0])
            s = parts[1].strip()
        hms = s.split(":")
        if len(hms) == 3:
            h, m, sec = int(hms[0]), int(hms[1]), int(float(hms[2]))
            return extra_days * 86400 + h * 3600 + m * 60 + sec
        return None
    except Exception:
        return None

time_to_seconds_udf = F.udf(time_to_seconds, IntegerType())

print("Setup done — all UDFs registered")

In [0]:

# Goal: Confirm we can read the table and understand schema + size.
 
df_bronze = spark.read.table("marathos.bronze.races")
 
print("Schema:")
df_bronze.printSchema()
 
print(f"\nRow count: {df_bronze.count():,}")
display(df_bronze)

In [0]:

# Goal: Understand how much data is missing and what we are working with.
 
# Nulls per column
total_rows = df_bronze.count()
for col in df_bronze.columns:
    null_cnt = df_bronze.filter(F.col(col).isNull()).count()
    pct = round(null_cnt / total_rows * 100, 2)
    print(f"{col:45} nulls: {null_cnt:>6}  ({pct}%)")
 

In [0]:
# Goal: See which unique event units exist and how common they are.
 
print("Unique event_distance_length values:")
(df_bronze
 .groupBy("event_distance_length")
 .count()
 .orderBy(F.desc("count"))
 .display()
)
 

In [0]:
# Goal: Add helper columns for event_unit and performance_unit
#       and see how many rows violate the unit logic.

df_with_units = (
    df_bronze
    .withColumn(
        "event_unit",
        F.when(F.col("event_distance_length").endswith("km"), "km")
         .when(F.col("event_distance_length").endswith("mi"), "mi")
         .when(F.col("event_distance_length").endswith("h"),  "h")
         .when(F.col("event_distance_length").endswith("d"),  "d")
         .otherwise(None)
    )
    .withColumn(
        "performance_unit",
        F.when(F.col("athlete_performance").contains(":"), "h")
         .otherwise("km")
    )
)

# Add validity flag
df_with_units = df_with_units.withColumn(
    "is_valid_combo",
    F.when(
        (F.col("event_unit").isin("km", "mi")) & (F.col("performance_unit") == "h"), True
    ).when(
        (F.col("event_unit") == "h") & (F.col("performance_unit") == "km"), True
    ).otherwise(False)
)

# Show unit combination distribution
print("event_unit vs performance_unit:")
(df_with_units
 .groupBy("event_unit", "performance_unit", "is_valid_combo")
 .count()
 .orderBy(F.desc("count"))
 .display()
)

# Count invalid rows
invalid_count = df_with_units.filter(F.col("is_valid_combo") == False).count()
print(f"Invalid rows: {invalid_count:,}")

In [0]:

# Rules:
#   1. event_unit = 'd' (days) -> remove
#   2. Invalid unit combination -> remove
#   3. Nulls in critical columns -> remove
 
rows_before = df_with_units.count()
 
df_clean = (
    df_with_units
    # Rule 1: remove days (d)
    .filter(F.col("event_unit") != "d")
    # Rule 2: remove invalid unit combinations
    .filter(F.col("is_valid_combo") == True)
    # Rule 3: remove rows missing athlete_performance or event_distance_length
    .filter(F.col("athlete_performance").isNotNull())
    .filter(F.col("event_distance_length").isNotNull())
    .filter(F.col("athlete_id").isNotNull())
    # Filter out rows where athlete_average_speed looks like a time string
    .filter(~F.col("athlete_average_speed").contains(":"))
    # Filter out unrealistic speeds (keep between 1 and 30 km/h)
    .filter(
        F.regexp_replace(F.col("athlete_average_speed"), "[^0-9.]", "").cast("float").between(1, 30)
        | F.col("athlete_average_speed").isNull()
    )

)
 




rows_after = df_clean.count()
print(f"Rows before cleaning : {rows_before:,}")
print(f"Rows after cleaning  : {rows_after:,}")
print(f"Rows removed         : {rows_before - rows_after:,}")
 

In [0]:

# Convert athlete_performance to appropriate type (fixed)

def time_to_seconds(time_str):
    if time_str is None: return None
    try:
        s = time_str.strip()
        extra_days = 0
        if "day" in s:
            parts = s.split(",")
            extra_days = int(parts[0].strip().split(" ")[0])
            s = parts[1].strip()
        hms = s.split(":")
        if len(hms) == 3:
            h, m, sec = int(hms[0]), int(hms[1]), int(float(hms[2]))
            return extra_days * 86400 + h * 3600 + m * 60 + sec
        return None
    except Exception:
        return None

time_to_seconds_udf = F.udf(time_to_seconds, IntegerType())

df_converted = (
    df_clean
    # Time performances (HH:MM:SS) -> seconds
    .withColumn(
        "performance_seconds",
        F.when(
            F.col("performance_unit") == "h",
            time_to_seconds_udf(F.col("athlete_performance"))
        ).otherwise(None)
    )
    # Distance performances -> strip " km" suffix then cast to float
    .withColumn(
        "performance_km",
        F.when(
            F.col("performance_unit") == "km",
            F.split(F.col("athlete_performance"), " ").getItem(0).cast("float")
        ).otherwise(None)
    )
)

# Sanity check
print("Sample of converted performances:")
display(
    df_converted
    .select("athlete_performance", "performance_unit", "performance_seconds", "performance_km")
    .limit(20)
)

null_sec = df_converted.filter(
    (F.col("performance_unit") == "h") & F.col("performance_seconds").isNull()
).count()
print(f"Failed time conversions (should be 0): {null_sec}")

dense_rank() räknar upp unika värden i bokstavsordning och ger varje ett ID


In [0]:
 

# event_id — based on event_name
w_event = Window.orderBy("event_name")
df_ids = df_converted.withColumn("event_id", F.dense_rank().over(w_event))

# athlete_key — based on athlete_id
w_athlete = Window.orderBy("athlete_id")
df_ids = df_ids.withColumn("athlete_key", F.dense_rank().over(w_athlete))

# year_id — based on year_of_event
w_year = Window.orderBy("year_of_event")
df_ids = df_ids.withColumn("year_id", F.dense_rank().over(w_year))

# Sanity checks
display(df_ids.select("event_name", "event_id").distinct().orderBy("event_id").limit(10))
display(df_ids.select("athlete_id", "athlete_key").distinct().orderBy("athlete_key").limit(10))
display(df_ids.select("year_of_event", "year_id").distinct().orderBy("year_id"))

In [0]:

 
# Clean remaining columns and fix data types


df_silver = (
    df_ids
    # Fix athlete_average_speed — split on space and take first part only
    .withColumn(
        "athlete_average_speed",
        F.split(F.col("athlete_average_speed"), " ").getItem(0).cast("float")
    )
    # Fix other data types
    .withColumn("athlete_year_of_birth",     F.col("athlete_year_of_birth").cast("int"))
    .withColumn("event_number_of_finishers", F.col("event_number_of_finishers").cast("int"))
    # Normalize text
    .withColumn("athlete_gender",  F.upper(F.trim(F.col("athlete_gender"))))
    .withColumn("athlete_country", F.upper(F.trim(F.col("athlete_country"))))
    # Derive age
    .withColumn(
        "athlete_age_at_event",
        F.col("year_of_event") - F.col("athlete_year_of_birth")
    )
    # Drop helper columns
    .drop("is_valid_combo", "event_unit", "performance_unit")
)

print(f"Total rows: {df_silver.count():,}")
display(df_silver.limit(20))

In [0]:

print(f"Total rows       : {df_silver.count():,}")
print(f"Unique events    : {df_silver.select('event_id').distinct().count():,}")
print(f"Unique athletes  : {df_silver.select('athlete_key').distinct().count():,}")
print(f"Unique years     : {df_silver.select('year_id').distinct().count():,}")
print(f"Null performance_seconds: {df_silver.filter(F.col('performance_seconds').isNull() & (F.col('performance_km').isNull())).count():,}")

display(df_silver.limit(20))

In [0]:
# ════════════════════════════════════════════════════════
# STEP 9 — Review final OBT
# ════════════════════════════════════════════════════════
 
print("Final schema:")
df_silver.printSchema()
 
print(f"\nRow count in Silver OBT: {df_silver.count():,}")
 
# Browse a sample
df_silver.display()
 
# Quick check on key columns
df_silver.select(
    "event_id", "event_name", "event_distance_length",
    "athlete_key", "athlete_id",
    "athlete_performance", "performance_seconds", "performance_km",
    "athlete_gender", "athlete_country", "year_of_event"
).limit(20).display()
 

In [0]:
# Investigate athlete_average_speed — using df_clean
display(
    df_clean
    .select("athlete_average_speed", "event_distance_length")
    .withColumn(
        "speed_float",
        F.regexp_replace(F.col("athlete_average_speed"), "[^0-9.]", "").cast("float")
    )
    .filter(F.col("speed_float").isNotNull())
    .orderBy(F.desc("speed_float"))
    .limit(20)
)

In [0]:
# Check distribution of speeds
display(
    df_clean
    .select(
        F.regexp_replace(F.col("athlete_average_speed"), "[^0-9.]", "").cast("float").alias("speed_float")
    )
    .filter(F.col("speed_float").isNotNull())
    .select(
        F.min("speed_float").alias("min_speed"),
        F.max("speed_float").alias("max_speed"),
        F.avg("speed_float").alias("avg_speed"),
        F.percentile_approx("speed_float", 0.99).alias("99th_percentile")
    )
)